In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import torch
import torch.nn as nn

In [3]:
import os
import shutil

src_root = r"/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray"
dst_root = "output"


def process_split(split_name):
    split_src = os.path.join(src_root, split_name)
    split_dst = os.path.join(dst_root, split_name)

    normal_dst = os.path.join(split_dst, "NORMAL")
    viral_dst = os.path.join(split_dst, "VIRAL_PNEUMONIA")
    bacterial_dst = os.path.join(split_dst, "BACTERIAL_PNEUMONIA")

    os.makedirs(normal_dst, exist_ok=True)
    os.makedirs(viral_dst, exist_ok=True)
    os.makedirs(bacterial_dst, exist_ok=True)

    # NORMAL
    normal_src = os.path.join(split_src, "NORMAL")
    for file in os.listdir(normal_src):
        src = os.path.join(normal_src, file)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(normal_dst, file))

    # PNEUMONIA
    pneumonia_src = os.path.join(split_src, "PNEUMONIA")
    for file in os.listdir(pneumonia_src):
        src = os.path.join(pneumonia_src, file)

        if not os.path.isfile(src):
            continue

        name = file.lower()

        if "virus" in name or "viral" in name:
            shutil.copy2(src, os.path.join(viral_dst, file))
        else:
            shutil.copy2(src, os.path.join(bacterial_dst, file))

    print(f"{split_name} done!")


# Process all splits
for split in ["train", "test", "val"]:
    process_split(split)

print("All Done!")

train done!
test done!
val done!
All Done!


In [32]:
class PatchEmbedding(nn.Module):

    def __init__(
        self,
        image_size=224,
        patch_size=16,
        in_channels=3,
        embed_dim=768
    ):
        super().__init__()

        self.num_patches = (image_size // patch_size) ** 2

        self.projection = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):

        # x = (B,3,224,224)

        x = self.projection(x)

        # (B,768,14,14)

        x = x.flatten(2)

        # (B,768,196)

        x = x.transpose(1,2)

        # (B,196,768)

        return x

In [33]:
class MLP(nn.Module):

    def __init__(self, embed_dim, hidden_dim, dropout=0.3):
        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(embed_dim, hidden_dim),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(hidden_dim, embed_dim),

            nn.Dropout(dropout)
        )

    def forward(self,x):
        return self.net(x)

In [62]:
class TransformerEncoder(nn.Module):

    def __init__(
        self,
        embed_dim=768,
        num_heads=12,
        mlp_ratio=4,
        dropout=0.1
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(embed_dim)

        self.attention = nn.MultiheadAttention(
            embed_dim,
            num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm2 = nn.LayerNorm(embed_dim)

        self.mlp = MLP(
            embed_dim,
            embed_dim * mlp_ratio,
            dropout
        )

    def forward(self,x):

        y = self.norm1(x)

        attn,_ = self.attention(
            y,
            y,
            y
        )
         

        x = x + attn

        y = self.norm2(x)

        x = x + self.mlp(y)

        return x

In [63]:
class VisionTransformer(nn.Module):

    def __init__(
        self,
        image_size=224,
        patch_size=16,
        in_channels=3,
        num_classes=1000,
        embed_dim=768,
        depth=2,
        num_heads=12
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(
            image_size,
            patch_size,
            in_channels,
            embed_dim
        )

        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(
            torch.randn(1,1,embed_dim)
        )

        self.position_embedding = nn.Parameter(
            torch.randn(1,num_patches+1,embed_dim)
        )

        self.dropout = nn.Dropout(0.3)

        self.encoder = nn.Sequential(
            *[
                TransformerEncoder(
                    embed_dim,
                    num_heads
                )
                for _ in range(depth)
            ]
        )

        self.norm = nn.LayerNorm(embed_dim)

        self.head = nn.Linear(
            embed_dim*2,
            num_classes
        )

    def forward(self,x):

        B = x.shape[0]

        x = self.patch_embed(x)

        cls = self.cls_token.expand(B,-1,-1)

        x = torch.cat((cls,x),dim=1)

        x = x + self.position_embedding

        x = self.dropout(x)

        x = self.encoder(x)

        x = self.norm(x)
        cls=x[:,0]
        mean=x[:,1:].mean(dim=1)
        feature=torch.cat([cls,mean],dim=1)
        

        

        output = self.head(feature)

        return output

In [64]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [65]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((224, 224)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(10),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05)
    ),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15
    ),

    transforms.ToTensor(),

    transforms.RandomErasing(
        p=0.2,
        scale=(0.02, 0.08),
        ratio=(0.3, 3.3)
    ),

    transforms.Normalize(
        mean=[0.5],
        std=[0.5]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5],
        std=[0.5]
    )
])

In [66]:
train_dataset = datasets.ImageFolder(
    "/kaggle/working/output/train",
    transform=train_transform
)
 

test_dataset = datasets.ImageFolder(
    "/kaggle/working/output/test",
    transform=test_transform
)

In [67]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

 

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [68]:
print(train_dataset.classes)

['BACTERIAL_PNEUMONIA', 'NORMAL', 'VIRAL_PNEUMONIA']


In [69]:
print(train_dataset.class_to_idx)

{'BACTERIAL_PNEUMONIA': 0, 'NORMAL': 1, 'VIRAL_PNEUMONIA': 2}


In [70]:
model = VisionTransformer(
    image_size=224,
    patch_size=16,
    num_classes=3
)

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

Using 2 GPUs


In [71]:
sum([p.numel() for p in model.parameters()])

14924547

In [72]:
criterion = nn.CrossEntropyLoss()

In [73]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [74]:
from tqdm.auto import tqdm

In [75]:
from tqdm.auto import tqdm

def train_one_epoch(model, loader):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc="Training", leave=False)

    for images, labels in pbar:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        # Live update
        pbar.set_postfix(
            loss=f"{total_loss/(pbar.n+1):.4f}",
            acc=f"{100*correct/total:.2f}%"
        )

    loss = total_loss / len(loader)
    acc = correct / total

    return loss, acc

In [76]:
def evaluate(model, loader):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            total_loss += loss.item()

            _, predicted = torch.max(outputs,1)

            correct += (predicted==labels).sum().item()

            total += labels.size(0)

    loss = total_loss / len(loader)

    acc = correct / total

    return loss, acc

In [ ]:
epochs = 5
best_model = float("inf")

for epoch in range(epochs):

    print(f"\nEpoch {epoch+1}/{epochs}")

    train_loss, train_acc = train_one_epoch(model, train_loader)

    val_loss, val_acc = evaluate(model, test_loader)

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Acc  : {train_acc:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val Acc    : {val_acc:.4f}")
    print("-"*40)

    if val_loss < best_model:
        best_model = val_loss
        torch.save(model.state_dict(), "vit_model.pth")
        print("✅ Best model saved!")


Epoch 1/5


Training:   0%|          | 0/163 [00:00<?, ?it/s]

In [36]:
print(torch.cuda.device_count())

2
